In [ ]:
!pip install -q pyreadstat

In [ ]:
import geopandas as gpd
import pandas as pd
from google.colab import drive
import pyreadstat

In [ ]:
#Spatial smoothing after aggregation (see below)
import libpysal
from esda.moran import Moran
from libpysal.weights import Queen
import numpy as np

Mount Google Colab

In [ ]:
drive.mount('/content/drive')

File Paths

In [ ]:
DHSclusters_path = "/content/drive/MyDrive/MYPROJECTFOLDER/ZWGE72FL.shp"
DHSbirth_path = "/content/drive/MyDrive/MYPROJECTFOLDER/ZWBirthRecode/ZWBR72FL.DTA"
districts_path = "/content/drive/MyDrive/MYPROJECTFOLDER/Zim_Fixed_Constituencies.shp"

Load Data

In [ ]:
clusters = gpd.read_file(DHSclusters_path)
districts = gpd.read_file(districts_path)

print(type(clusters))
print(type(districts))

print(clusters.columns)
print(districts.columns)

In [ ]:
br, meta = pyreadstat.read_dta(
    DHSbirth_path
)

print(type(br))
print(br.columns)

Safeguards

In [ ]:
assert isinstance(clusters, gpd.GeoDataFrame)
assert isinstance(districts, gpd.GeoDataFrame)

Check Geometry Types

In [ ]:
print("Cluster geometry types:")
print(clusters.geom_type.unique())

print("District geometry types:")
print(districts.geom_type.unique())

Reproject to Common CRS

In [ ]:
clusters = clusters.to_crs("ESRI:102022")
districts = districts.to_crs("ESRI:102022")

Fix Geometries

In [ ]:
clusters = clusters[clusters.is_valid]

districts["geometry"] = districts.buffer(0)

Join Clusters & Districts

In [ ]:
clusters_joined = gpd.sjoin(
    clusters,
    districts,
    how="left",
    predicate="within"
)

print(clusters_joined.columns)

In [ ]:
cluster_districts = clusters_joined[
    ["DHSCLUST", "Cons_name"]
].copy()

Death Indicator

In [ ]:
br["under5_dead"] = (
    (br["b5"] == 0) &
    (br["b7"] < 60)
).astype(int)

Weights

In [ ]:
br["weight"] = br["v005"] / 1000000

Merge on Cluster ID

In [ ]:
br = br.merge(
    cluster_districts,
    left_on="v001",
    right_on="DHSCLUST",
    how="left"
)

Weighted Aggregation

In [ ]:
br["weighted_death"] = br["under5_dead"] * br["weight"]

In [ ]:
district_mortality = (
    br.groupby("Cons_name")
      .agg(
          weighted_births=("weight", "sum"),
          weighted_deaths=("weighted_death", "sum")
      )
      .reset_index()
)

Compute Mortality Rate

In [ ]:
district_mortality["u5mr"] = (
    district_mortality["weighted_deaths"]
    /
    district_mortality["weighted_births"]
) * 1000

In [ ]:
print(district_mortality["u5mr"].describe())

Merge Mortality Rates Back to Districts

In [ ]:
districts = districts.merge(
    district_mortality,
    on="Cons_name",
    how="left"
)

In [ ]:
#Identify missing values
districts["u5mr_missing"] = districts["u5mr"].isna().astype(int)

Missing Values

In [ ]:
#Check for amount of missing values
print(districts["u5mr"].isna().sum())
print(len(districts))

In [ ]:
print(
    districts.loc[
        districts["u5mr"].isna(),
        "Cons_name"
    ].tolist()
)

Spatial Smoothing

In [ ]:
w = Queen.from_dataframe(districts, use_index=True)
w.transform = "R"

districts["u5mr_imputed"] = districts["u5mr"].copy()

u5mr = districts["u5mr_imputed"].values

for i in range(len(u5mr)):

    if np.isnan(u5mr[i]):

        neighbors = w.neighbors[i]

        neighbor_vals = [
            u5mr[j]
            for j in neighbors
            if not np.isnan(u5mr[j])
        ]

        if len(neighbor_vals) > 0:
            u5mr[i] = np.mean(neighbor_vals)

districts["u5mr_imputed"] = u5mr

In [ ]:
#Check remaining missings
print(districts["u5mr_imputed"].isna().sum())

In [ ]:
#Spatial lags
districts["u5mr_smooth"] = (
    libpysal.weights.lag_spatial(
        w,
        districts["u5mr_imputed"]
    )
)

Sanity Check

In [ ]:
print(
    districts[
        [
            "Cons_name",
            "weighted_births",
            "weighted_deaths",
            "u5mr",
            "u5mr_missing",
            "u5mr_imputed",
            "u5mr_smooth"
        ]
    ].head()
)

Export JSON

In [ ]:
output_json = (
    "/content/drive/MyDrive/MYPROJECTFOLDER/mortality_rate.json"
)

df = districts.drop(columns="geometry")

df.to_json(output_json, orient="records")

Export GeoJSON

In [ ]:
output_geojson = (
    "/content/drive/MyDrive/MYPROJECTFOLDER/mortality_rate.geojson"
)

districts.to_file(
    output_geojson,
    driver="GeoJSON"
)

print("Done.")